# Delta fare finder

delta.com's *My dates are flexible* grid only prices one origin → one destination at a time. This notebook asks the same API for **every home airport × European airport**, then ranks:

* **Round trips** (e.g. CLT ⇄ MUC, then a train/budget flight to Italy and back),
* **Open-jaw / multi-city** trips (fly into Germany, home from Italy, or the reverse),
* **Two one-way tickets**,

by what the whole trip costs, including your estimate for getting between countries once you're in Europe.

**How to use:** edit the settings in step 1, then *Runtime → Run all*. A full run is ~170 searches (8–10 minutes).

**If step 3 says Delta refused the request** (HTTP 403 or network errors): Delta often blocks cloud servers like Colab. Run `delta_fare_finder.js` in your browser's DevTools console on delta.com instead (see the README). It prints the same tables and downloads a `delta-fares-*.json` file you can analyse in step 5 below.

In [ ]:
# 1. Settings
CONFIG = {
    # Airports you can leave from and come home to.
    'homeAirports': ['CAE', 'CLT', 'GSP', 'CHS'],

    # Getting to the airport. Every trip's est. total adds gas for the drive there and back
    # plus parking for each day you're away (nights + 1). Miles are one way from Columbia, SC;
    # parking is each airport's cheapest on-airport long-term lot (airport websites, Oct 2026).
    # Edit for where you live. dropOff: True = someone drives you (gas for two round trips, no parking).
    'drive': {
        'gasPerGallon': 3.00,
        'milesPerGallon': 28,
        'dropOff': False,
        'airports': {
            'CAE': {'miles': 10, 'parkingPerDay': 12},
            'CLT': {'miles': 95, 'parkingPerDay': 14},
            'GSP': {'miles': 95, 'parkingPerDay': 10},
            'CHS': {'miles': 115, 'parkingPerDay': 13},
            'AGS': {'miles': 75, 'parkingPerDay': 10},
            'SAV': {'miles': 160, 'parkingPerDay': 7},
            'ATL': {'miles': 215, 'parkingPerDay': 14},
            'MYR': {'miles': 150, 'parkingPerDay': 12},
            'AVL': {'miles': 160, 'parkingPerDay': 12},
            'RDU': {'miles': 225, 'parkingPerDay': 10},
        },
    },
    # Anything else an airport costs you per trip, e.g. {'ATL': 120} for a hotel the night before.
    'homeAirportExtraCost': {},

    # European airports to price, grouped by country.
    'destinations': {
        'Germany': ['FRA', 'MUC', 'BER', 'DUS', 'HAM', 'STR'],
        'Italy': ['FCO', 'MXP', 'VCE', 'NAP', 'BLQ', 'FLR'],
    },
    # Countries the trip has to include, and your estimate (per person) for one train /
    # budget-airline hop between them. A round trip into Germany needs two hops
    # (Germany -> Italy -> back to Germany); flying into Germany and home from Italy needs one.
    'visitCountries': ['Germany', 'Italy'],
    'hopCost': 75,

    # Date ranges, inclusive. Each 7-day block of departures x 7-day block of returns
    # is one search per route, so wider ranges multiply the number of searches.
    'departDates': ['2026-12-14', '2026-12-20'],
    'returnDates': ['2026-12-22', '2026-12-28'],
    'minNights': 1,
    'maxNights': 30,

    'adults': 1,
    'nonStopOnly': False,
    'brand': 'MAIN',  # delta.com's default "Best fares for"; its grid already includes Basic Economy fares

    'searchRoundTrips': True,
    'searchOpenJaw': True,   # multi-city: into one country, home from another
    'searchOneWays': True,   # two separate one-way tickets
    # Open-jaw and one-way searches only use the N cheapest airports per country
    # from the round-trip scan, to keep the number of searches down.
    'gatewaysPerCountry': 3,
    # Let one-way pairs leave from one home airport and return to another.
    'mixHomeAirports': False,

    'delayMs': [1200, 2500],  # pause between searches; keep it polite or Delta may block you for a while
    'maxRequests': 400,
    'cacheHours': 6,
    'showTop': 25,
}

In [ ]:
# 2. Search engine (just run this cell)
import json
import os
import random
import time
import uuid
from datetime import date, timedelta
from urllib.parse import quote

import pandas as pd
import requests
from IPython.display import display

OFFERS_URL = 'https://offer-api-prd.delta.com/prd/rm-offer-gql'
CALLBACK_URL = 'https://offer-api-prd.delta.com/prd/callback/'
CALLBACK_CODE = 'ROF5004_PXY1007'  # delta.com's code for "still working, poll the callback URL"
CACHE_FILE = 'delta_fare_cache.json'

# The flexible-dates query delta.com sends, verbatim.
QUERY = """query ($offerSearchCriteria: OfferSearchCriteriaInput!) {
  gqlSearchOffers(offerSearchCriteria: $offerSearchCriteria) {
    offerResponseId
    gqlOffersSets {
      offers {
        offerId
        additionalOfferProperties {
          offered
          soldOut
          lowestFare
          totalTripStopCnt
          discountAvailable
        }
        offerPricing {
          discountsApplied {
            code
            pct
          }
          originalTotalAmt {
            currencyEquivalentPrice {
              currencyAmt
              roundedCurrencyAmt
              formattedCurrencyAmt
            }
            milesEquivalentPrice {
              mileCnt
            }
          }
          totalAmt {
            currencyEquivalentPrice {
              currencyAmt
              roundedCurrencyAmt
              formattedCurrencyAmt
            }
            milesEquivalentPrice {
              mileCnt
            }
          }
          promotionalPrices {
            code
            pct
            price {
              milesEquivalentPrice {
                mileCnt
              }
              currencyEquivalentPrice {
                currencyAmt
                roundedCurrencyAmt
                formattedCurrencyAmt
              }
            }
          }
        }
      }
      itineraryDepartureDate
    }
    offerDataList {
      pricingOptions {
        pricingOptionDetail {
          currencyCode
        }
      }
      flexReturnDates
      responseProperties {
        tripTypeText
        discountInfo {
          discountPct
          discountTypeCode
          nonDiscountedOffersAvailable
        }
        promotionsInfo {
          promotionalCode
          promotionalPct
        }
      }
    }
  }
}"""


class Halt(Exception):
    """Stop searching, but still rank what was found so far."""


def add_days(day, n):
    return (date.fromisoformat(day) + timedelta(days=n)).isoformat()


def nights_between(a, b):
    return (date.fromisoformat(b) - date.fromisoformat(a)).days


def block_centers(date_range):
    """Each search returns a 7x7 grid: 3 days either side of the date sent."""
    start, end = date_range
    centers = []
    while start <= end:
        centers.append(add_days(start, 3))
        start = add_days(start, 7)
    return centers


def block_pairs(cfg):
    """Departure/return blocks with at least one date combination inside the nights limits."""
    return [(o, r) for o in block_centers(cfg['departDates']) for r in block_centers(cfg['returnDates'])
            if nights_between(o, r) + 6 >= cfg['minNights'] and nights_between(o, r) - 6 <= cfg['maxNights']]


def leg(origin, destination, center):
    return {
        'departureLocalTs': f'{center}T00:00:00',
        'destinations': [{'airportCode': destination}],
        'origins': [{'airportCode': origin}],
        'calenderDateRequest': {'daysBeforeCnt': 3, 'daysAfterCnt': 3},  # Delta's spelling
    }


def search_criteria(legs, cfg):
    return {
        'productGroups': [{'productCategoryCode': 'FLIGHTS'}],
        'customers': [{'passengerTypeCode': 'ADT', 'passengerId': str(i + 1)} for i in range(cfg['adults'])],
        'offersCriteria': {
            'resultsPageNum': 4,
            'pricingCriteria': {'priceableIn': ['CURRENCY']},
            'preferences': {'nonStopOnly': cfg['nonStopOnly'], 'refundableOnly': False, 'excludeBrandTypes': []},
            'flightRequestCriteria': {'sortByBrandId': cfg['brand'], 'searchOriginDestination': legs},
        },
    }


def error_info(error):
    """Mirrors delta.com's error mapping: '<code>_<moreInfo code>' plus callback details."""
    error = error or {}
    details = (error.get('extensions') or {}).get('errors') or {}
    code = details.get('code', '')
    more = (details.get('moreInfo') or [{}])[0] or {}
    if more.get('code'):
        code = f"{code}_{more['code']}" if code else more['code']
    return {
        'code': code,
        'callbackId': more.get('callbackId', ''),
        'callbackInSecs': float(more.get('callbackInSecs') or 0),
        'message': error.get('message', ''),
    }


def parse_grid(offers):
    """Reads the grid the way delta.com does: each offer set is one outbound date. In a
    round-trip (or multi-city) grid its offers line up with flexReturnDates; in a one-way
    grid the fare for set i is offers[i]. Returns ([out, ret, price, stops] cells, currency)."""
    props = offers.get('offerDataList') or {}
    currency = (((props.get('pricingOptions') or [{}])[0] or {}).get('pricingOptionDetail') or {}).get('currencyCode')
    one_way = (props.get('responseProperties') or {}).get('tripTypeText') == 'ONE_WAY'
    return_dates = props.get('flexReturnDates') or []
    cells = []

    def add(out, ret, offer):
        if not offer:
            return
        p = offer.get('additionalOfferProperties') or {}
        pricing = (offer.get('offerPricing') or [{}])[0] or {}
        amount = ((pricing.get('totalAmt') or {}).get('currencyEquivalentPrice') or {}).get('currencyAmt')
        if amount is None or p.get('soldOut') or p.get('offered') is False:
            return
        cells.append([out, ret, amount, p.get('totalTripStopCnt')])

    for i, offer_set in enumerate(offers.get('gqlOffersSets') or []):
        found = offer_set.get('offers') or []
        out = offer_set.get('itineraryDepartureDate')
        if one_way:
            add(out, None, found[i] if i < len(found) else (found[0] if len(found) == 1 else None))
        else:
            for j, offer in enumerate(found):
                add(out, return_dates[j] if j < len(return_dates) else None, offer)
    return cells, currency


class DeltaFareSearch:
    """Sends flexible-dates searches the way delta.com does for a guest, with a disk cache."""

    def __init__(self, cfg, cache_file=CACHE_FILE):
        self.cfg = cfg
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/155.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/plain, */*',
            'Accept-Language': 'en-US,en;q=0.9',
            'Origin': 'https://www.delta.com',
            'Referer': 'https://www.delta.com/',
        })
        self.transaction_prefix = str(uuid.uuid4())
        self.requests = 0
        self.currency = 'USD'
        self.cache_file = cache_file
        self.cache = {}
        try:
            with open(cache_file) as f:
                saved = json.load(f)
            cutoff = time.time() - cfg['cacheHours'] * 3600
            self.cache = {k: v for k, v in saved.items() if v.get('t', 0) > cutoff}
        except (OSError, ValueError):
            pass

    def _http(self, method, url, **kwargs):
        for attempt in range(4):
            try:
                res = self.session.request(method, url, timeout=45, **kwargs)
            except requests.RequestException as e:
                if attempt >= 2:
                    raise Halt(f'network error ({e}); Delta may be blocking this machine') from e
                time.sleep(5 * (attempt + 1))
                continue
            if res.status_code in (401, 403):
                raise Halt(f'HTTP {res.status_code}: Delta refused the request. It often blocks cloud servers '
                           'like Colab; run delta_fare_finder.js in your browser console instead (see step 5)')
            if res.status_code in (429, 502, 503, 504):
                if attempt >= 3:
                    raise Halt(f'HTTP {res.status_code} after {attempt + 1} tries')
                wait = 15 * 2 ** attempt
                print(f'HTTP {res.status_code}; waiting {wait}s before retrying')
                time.sleep(wait)
                continue
            try:
                data = res.json()
            except ValueError:
                data = None
            return data if isinstance(data, dict) else {'errors': [{'message': f'HTTP {res.status_code}'}]}

    def _fetch_grid(self, legs):
        headers = {
            'Authorization': 'GUEST',
            'Content-Type': 'application/json',
            'TransactionId': f'{self.transaction_prefix}_{int(time.time() * 1000)}',
            'applicationId': 'DC',
            'x-app-type': 'dcom-shop',
            'x-app-route': 'dates',
            'channelId': 'DCOM',
            'Airline': 'DL',
        }
        body = {'variables': {'offerSearchCriteria': search_criteria(legs, self.cfg)}, 'query': QUERY}
        data = self._http('POST', OFFERS_URL, data=json.dumps(body), headers=headers)
        polls = 0
        while True:
            offers = (data.get('data') or {}).get('gqlSearchOffers')
            if offers:
                cells, currency = parse_grid(offers)
                self.currency = currency or self.currency
                return {'cells': cells}
            err = error_info((data.get('errors') or [None])[0])
            if err['code'] != CALLBACK_CODE or not err['callbackId']:
                return {'cells': [], 'error': err['code'] or err['message'] or 'empty response'}
            if polls == 10:
                return {'cells': [], 'error': 'gave up waiting for callback', 'transient': True}
            polls += 1
            time.sleep(max(1, err['callbackInSecs']))
            data = self._http('GET', CALLBACK_URL + quote(err['callbackId'], safe=''), headers={'Authorization': 'GUEST'})

    def grid(self, legs):
        key = json.dumps([legs, self.cfg['adults'], self.cfg['brand'], self.cfg['nonStopOnly']], sort_keys=True)
        if key in self.cache:
            self.currency = self.cache[key].get('currency') or self.currency
            return {**self.cache[key], 'cached': True}
        if self.requests >= self.cfg['maxRequests']:
            raise Halt(f"reached maxRequests ({self.cfg['maxRequests']})")
        if self.requests:
            time.sleep(random.uniform(*self.cfg['delayMs']) / 1000)
        self.requests += 1
        result = self._fetch_grid(legs)
        if not result.get('transient'):
            self.cache[key] = {'t': time.time(), 'currency': self.currency, 'cells': result['cells'], 'error': result.get('error')}
            try:
                with open(self.cache_file, 'w') as f:
                    json.dump(self.cache, f)
            except OSError:
                pass  # carry on without caching
        return result


def money_formatter(currency):
    return (lambda n: f'${n:,.0f}') if currency == 'USD' else (lambda n: f'{n:,.0f} {currency}')


def pick_gateways(rows, cfg):
    """The N cheapest airports per country from the round-trip scan (or the first N listed)."""
    best = {}
    for r in rows:
        if r['kind'] == 'RT':
            best[r['to']] = min(best.get(r['to'], float('inf')), r['price'])
    picked = {}
    for country, airports in cfg['destinations'].items():
        priced = sorted((a for a in airports if a in best), key=best.get)
        picked[country] = (priced or airports)[:cfg['gatewaysPerCountry']]
    return picked


def plan(cfg, gateways):
    homes = cfg['homeAirports']
    pairs = block_pairs(cfg)
    multi = len(pairs) > 1
    trip_countries = [c for c in cfg['visitCountries'] if c in cfg['destinations']]

    def dates(o, r):
        return f' [{add_days(o, -3)}…{add_days(o, 3)} / {add_days(r, -3)}…{add_days(r, 3)}]' if multi else ''

    def block(center, centers):
        return f' [{add_days(center, -3)}…{add_days(center, 3)}]' if len(centers) > 1 else ''

    round_trips = [
        {'legs': [leg(h, d, o), leg(d, h, r)], 'row': {'kind': 'RT', 'from': h, 'to': d, 'from2': d, 'to2': h},
         'label': f'{h} ⇄ {d}{dates(o, r)}'}
        for h in homes for airports in cfg['destinations'].values() for d in airports for o, r in pairs]
    open_jaws = [
        {'legs': [leg(h, x, o), leg(y, h, r)], 'row': {'kind': 'OJ', 'from': h, 'to': x, 'from2': y, 'to2': h},
         'label': f'{h} → {x} … {y} → {h}{dates(o, r)}'}
        for in_c in trip_countries for out_c in trip_countries if out_c != in_c
        for h in homes for x in gateways[in_c] for y in gateways[out_c] for o, r in pairs]
    airports = [a for lst in gateways.values() for a in lst]
    out_centers, ret_centers = block_centers(cfg['departDates']), block_centers(cfg['returnDates'])
    one_ways = [
        {'legs': [leg(h, x, o)], 'row': {'kind': 'OW-out', 'from': h, 'to': x, 'from2': None, 'to2': None},
         'label': f'{h} → {x} one-way{block(o, out_centers)}'}
        for h in homes for x in airports for o in out_centers] + [
        {'legs': [leg(x, h, r)], 'row': {'kind': 'OW-back', 'from': x, 'to': h, 'from2': None, 'to2': None},
         'label': f'{x} → {h} one-way{block(r, ret_centers)}'}
        for x in airports for h in homes for r in ret_centers]
    return {'RT': round_trips, 'OJ': open_jaws, 'OW': one_ways}


def run_search(cfg, cache_file=CACHE_FILE):
    """Runs every search in cfg. Returns (rows, currency); rows are individual fares."""
    searcher = DeltaFareSearch(cfg, cache_file)
    money = money_formatter('USD')
    trip_countries = [c for c in cfg['visitCountries'] if c in cfg['destinations']]
    first_n = {c: a[:cfg['gatewaysPerCountry']] for c, a in cfg['destinations'].items()}
    estimate = plan(cfg, first_n)
    planned = ((len(estimate['RT']) if cfg['searchRoundTrips'] else 0)
               + (len(estimate['OJ']) if cfg['searchOpenJaw'] else 0)
               + (len(estimate['OW']) if cfg['searchOneWays'] else 0))
    seconds = sum(cfg['delayMs']) / 2000 + 1.5
    print(f"Up to {planned} searches (~{-(-planned * seconds // 60):.0f} min if none are cached). "
          f"Departing {cfg['departDates'][0]}–{cfg['departDates'][1]}, returning {cfg['returnDates'][0]}–{cfg['returnDates'][1]}.")
    if planned > cfg['maxRequests']:
        print(f"That's more than maxRequests ({cfg['maxRequests']}). Narrow the dates/airports or raise maxRequests.")
        return [], searcher.currency

    rows = []

    def run_tasks(name, tasks, bail_if_unsupported=False):
        print(f'\n{name}: {len(tasks)} searches')
        for n, task in enumerate(tasks):
            result = searcher.grid(task['legs'])
            task['result'] = result
            for out, ret, price, stops in result['cells']:
                rows.append({**task['row'], 'out': out, 'ret': ret, 'price': price, 'stops': stops})
            prices = [c[2] for c in result['cells']]
            outcome = f'from {money(min(prices))}' if prices else 'no fares' + (f" ({result['error']})" if result.get('error') else '')
            print(f"[{name} {n + 1}/{len(tasks)}] {task['label']}: {outcome}{' (cached)' if result.get('cached') else ''}")
            if bail_if_unsupported and n == 2 and all(t['result'].get('error') for t in tasks[:3]):
                print(f"{name}: the first 3 searches all failed, so Delta's flexible-dates API probably doesn't price these. Skipping the rest.")
                return

    try:
        if cfg['searchRoundTrips']:
            run_tasks('Round trips', plan(cfg, first_n)['RT'])
        gateways = pick_gateways(rows, cfg)
        if cfg['searchOpenJaw'] and len(trip_countries) > 1:
            print(f'\nOpen-jaw gateways: {gateways}')
            run_tasks('Open-jaw', plan(cfg, gateways)['OJ'], bail_if_unsupported=True)
        if cfg['searchOneWays']:
            run_tasks('One-ways', plan(cfg, gateways)['OW'])
    except (Halt, KeyboardInterrupt) as e:
        print(f'\nStopped early: {e or "interrupted"}. Ranking what was found so far; '
              're-run later to continue (finished searches are cached).')

    with open('delta_fares.json', 'w') as f:
        json.dump({'config': cfg, 'currency': searcher.currency, 'rows': rows}, f)
    print(f'\nDone: {searcher.requests} searches sent, {len(rows)} fares (saved to delta_fares.json).')
    return rows, searcher.currency


COLUMNS = ['strategy', 'route', 'from', 'to', 'from2', 'to2', 'out', 'ret', 'nights', 'stops', 'airfare', 'driveAndPark', 'hops', 'total']


def ground_cost(cfg, airport, nights):
    """Gas to the airport and back plus parking for every day away (or gas for two round trips
    when someone drops you off), plus anything in homeAirportExtraCost."""
    drive = cfg.get('drive') or {}
    a = (drive.get('airports') or {}).get(airport)
    extra = (cfg.get('homeAirportExtraCost') or {}).get(airport, 0)
    if not a:
        return extra
    gas_per_mile = (drive.get('gasPerGallon') or 0) / (drive.get('milesPerGallon') or 25)
    if drive.get('dropOff'):
        cost = 4 * a['miles'] * gas_per_mile
    else:
        cost = 2 * a['miles'] * gas_per_mile + (a.get('parkingPerDay') or 0) * (nights + 1)
    return round(cost + extra)


def build_trips(rows, cfg):
    """Every round trip, open-jaw and one-way pair that fits the dates, cheapest first."""
    # Countries of every airport: destinations, the console script's extra airports, and
    # (in its exports) airports its nearby-airports search turned up.
    country_of = {a: c for c, lst in {**(cfg.get('moreAirports') or {}), **cfg['destinations']}.items() for a in lst}
    country_of.update(cfg.get('countryOf') or {})
    known = set(cfg['destinations']) | set(cfg.get('moreAirports') or {})
    trip_countries = [c for c in cfg['visitCountries'] if c in known]

    def hops_needed(in_country, out_country):
        # Start in the arrival airport's country, pass through every country in
        # visitCountries, finish in the departure airport's country.
        countries = set(trip_countries) | {in_country, out_country}
        if in_country != out_country:
            return len(countries) - 1
        return len(countries) if len(countries) > 1 else 0

    def dates_ok(out, ret):
        if not out or not ret:
            return False
        n = nights_between(out, ret)
        return (cfg['departDates'][0] <= out <= cfg['departDates'][1]
                and cfg['returnDates'][0] <= ret <= cfg['returnDates'][1]
                and cfg['minNights'] <= n <= cfg['maxNights'])

    def make(strategy, frm, to, frm2, to2, out, ret, airfare, stops):
        hops = hops_needed(country_of.get(to), country_of.get(frm2))
        nights = nights_between(out, ret)
        # Leaving from one airport and coming home to another (mixHomeAirports): average the two.
        ground = (ground_cost(cfg, frm, nights) + ground_cost(cfg, to2, nights)) / 2
        route = f'{frm} ⇄ {to}' if frm == to2 and to == frm2 else f'{frm} → {to} … {frm2} → {to2}'
        return [strategy, route, frm, to, frm2, to2, out, ret, nights, stops, airfare, ground, hops,
                airfare + ground + hops * cfg['hopCost'] * cfg['adults']]

    stop_label = lambda s: '?' if s is None else str(s)
    trips = [make('Round trip' if r['kind'] == 'RT' else 'Open-jaw', r['from'], r['to'], r['from2'], r['to2'],
                  r['out'], r['ret'], r['price'], stop_label(r['stops']))
             for r in rows if r['kind'] in ('RT', 'OJ') and dates_ok(r['out'], r['ret'])]
    outbound = [r for r in rows if r['kind'] == 'OW-out']
    inbound = [r for r in rows if r['kind'] == 'OW-back']
    trips += [make('Two one-ways', a['from'], a['to'], b['from'], b['to'], a['out'], b['out'], a['price'] + b['price'],
                   f"{stop_label(a['stops'])}+{stop_label(b['stops'])}")
              for a in outbound for b in inbound
              if (cfg['mixHomeAirports'] or b['to'] == a['from']) and dates_ok(a['out'], b['out'])]
    df = pd.DataFrame(trips, columns=COLUMNS)
    return df.sort_values(['total', 'airfare'], kind='stable').reset_index(drop=True)


def how_to_book(t):
    if t['strategy'] == 'Round trip':
        return f"Round Trip {t['from']} → {t['to']}, depart {t['out']}, return {t['ret']}"
    if t['strategy'] == 'Open-jaw':
        return f"Multi-City: {t['from']} → {t['to']} on {t['out']}, then {t['from2']} → {t['to2']} on {t['ret']}"
    return f"Two One-Way tickets: {t['from']} → {t['to']} on {t['out']}, and {t['from2']} → {t['to2']} on {t['ret']}"


def show_results(rows, cfg, currency='USD'):
    """Prints the ranking tables, saves delta_trips.csv and returns every trip as a DataFrame."""
    trips = build_trips(rows, cfg)
    if not rows:
        print('No fares to rank: step 3 found nothing. If it said Delta refused the request, '
              'run the console script and upload its JSON in step 5.')
        return trips
    if trips.empty:
        print('No fares matched. Check the dates (future, within the ranges) and the nights limits.')
        return trips
    money = money_formatter(currency)
    best = trips.drop_duplicates(['strategy', 'from', 'to', 'from2', 'to2']).reset_index(drop=True)
    known = (cfg.get('drive') or {}).get('airports') or {}
    missing = sorted({a for a in pd.concat([trips['from'], trips['to2']]).dropna() if a not in known})
    if known and missing:
        print(f"No drive/parking figures for {', '.join(missing)}: their trips are ranked as if getting there were free. "
              "Add them to CONFIG['drive']['airports'].\n")

    def view(df):
        return (df.assign(airfare=df['airfare'].map(money), driveAndPark=df['driveAndPark'].map(money), total=df['total'].map(money))
                [['strategy', 'route', 'out', 'ret', 'nights', 'stops', 'airfare', 'driveAndPark', 'hops', 'total']]
                .rename(columns={'out': 'depart', 'ret': 'return', 'driveAndPark': 'drive + park', 'hops': 'Europe hops', 'total': 'est. total'}))

    countries = ' and '.join(c for c in cfg['visitCountries'] if c in set(cfg['destinations']) | set(cfg.get('moreAirports') or {}))
    print(f"Cheapest by strategy (est. total = airfare + drive & parking + {money(cfg['hopCost'])} per person per hop between {countries}):")
    display(view(best.drop_duplicates('strategy')).reset_index(drop=True))
    print(f"\nTop {cfg['showTop']} routes (best dates for each):")
    display(view(best.head(cfg['showTop'])))

    round_trips = trips[trips['strategy'] == 'Round trip']
    if not round_trips.empty:
        print('\nCheapest round-trip airfare, home airport × destination:')
        matrix = round_trips.pivot_table(index='from', columns='to', values='airfare', aggfunc='min')
        matrix = matrix.reindex(index=cfg['homeAirports'], columns=[a for lst in cfg['destinations'].values() for a in lst])
        display(matrix.apply(lambda col: col.map(lambda v: '—' if pd.isna(v) else money(v))))

    print('\nHow to book the top 5 on delta.com:')
    for i, t in best.head(5).iterrows():
        print(f"{i + 1}. {money(t['total'])} est. ({money(t['airfare'])} airfare): {how_to_book(t)}")
    print('Prices are for the whole party, before seat/bag fees, and can change by the minute. '
          'Search the exact dates on delta.com to confirm.')
    trips.to_csv('delta_trips.csv', index=False)
    print('\nEvery combination saved to delta_trips.csv.')
    return trips


In [ ]:
# 3. Search delta.com (re-running reuses grids fetched in the last few hours)
rows, currency = run_search(CONFIG)

In [ ]:
# 4. Results
trips = show_results(rows, CONFIG, currency)

In [ ]:
# Optional: download every combination as a CSV
if not os.path.exists('delta_trips.csv'):
    print('Nothing to download yet: step 4 had no fares to rank (see step 5 if Delta refused the search).')
else:
    try:
        from google.colab import files
        files.download('delta_trips.csv')
    except ImportError:
        print('Not on Colab: the CSV is saved as delta_trips.csv next to this notebook.')

## 5. Analyse results from the console script

If Delta blocked Colab, run `delta_fare_finder.js` in the DevTools console on a delta.com tab. When it finishes it downloads `delta-fares-<date>.json`. Run steps 1–2 above, then this cell, and pick that file. The JSON's search settings are used, with your step-1 values for `hopCost`, `drive`, `homeAirportExtraCost`, `minNights`/`maxNights`, `mixHomeAirports` and `showTop`, so you can re-rank without searching again. It ranks the fares on your dates; fares the console found for other weeks are in the console output and its CSV.

In [ ]:
# 5. Upload delta-fares-*.json from the console script.
# Runs by itself when step 3 found nothing (e.g. Delta blocked Colab);
# set ANALYSE_EXPORT = True to analyse a console export anyway.
ANALYSE_EXPORT = False

if ANALYSE_EXPORT or not globals().get('rows'):
    try:
        from google.colab import files
        uploaded = files.upload()
        export = json.loads(next(iter(uploaded.values())))
    except ImportError:  # not on Colab: point this at the file instead
        with open('delta-fares.json') as f:
            export = json.load(f)

    RERANK_KEYS = ['hopCost', 'drive', 'homeAirportExtraCost', 'minNights', 'maxNights', 'mixHomeAirports', 'showTop']
    export_config = {**export['config'], **{k: CONFIG[k] for k in RERANK_KEYS}, 'countryOf': export.get('countryOf') or {}}
    trips = show_results(export['rows'], export_config, export.get('currency', 'USD'))
else:
    print('Step 3 found fares, so skipping. Set ANALYSE_EXPORT = True to analyse a console export instead.')